# XE4 1D FIR via row-copy

A fully worked, cell-by-cell functional port of the sycl-tla example [`examples/cute/tutorial/xe4/api_amma_row_copy_fir_1d_xe4.cpp`](../../examples/cute/tutorial/xe4/api_amma_row_copy_fir_1d_xe4.cpp ), runnable on the CPU.

A 1D FIR filter expressed as row copies + tiny dot-product MMAs.

Every cell performs one operation on small concrete data and shows the matching Xe layout. Run them top to bottom. (A runnable script version lives in `api_amma_row_copy_fir_1d_xe4.py`.)

## Setup — data + a tiny layout printer

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True, linewidth=120)
from tensor_layouts import Layout, size
from tensor_layouts.analysis import is_bijective
from tensor_layouts.atoms_xe_common import make_slm_layout_elem, sizeof_bits

def show_layout(layout, n_rows, n_cols, rl="m", cl="k", max_r=8, max_c=8):
    "Print coord -> memory offset for a rank-2 layout (truncated)."
    R, C = min(n_rows, max_r), min(n_cols, max_c)
    print("      " + "".join((cl + str(j)).ljust(5) for j in range(C)) + (" ..." if C < n_cols else ""))
    for i in range(R):
        print((" " + rl + str(i)).ljust(6) + "".join(str(layout(i, j)).ljust(5) for j in range(C))
              + (" ..." if C < n_cols else ""))
    if R < n_rows:
        print("  ...  (%dx%d total)" % (n_rows, n_cols))

M, N, K = 32, 32, 16          # A rows, B rows, contraction
rng = np.random.default_rng(0)
A = rng.integers(-2, 3, size=(M, K)).astype(np.float32)   # A  (M x K)
B = rng.integers(-2, 3, size=(N, K)).astype(np.float32)   # B  (N x K), used as B^T
print("A", A.shape, " B", B.shape)
print("A[:4]:\n", A[:4])

A (32, 16)  B (32, 16)
A[:4]:
 [[ 2.  1.  0. -1. -1. -2. -2. -2. -2.  2.  1.  2.  0.  1.  2.  1.]
 [ 1.  0.  0.  2. -1.  2.  1. -2. -1.  2.  0. -2.  1.  1.  2. -2.]
 [-2.  2. -2.  0. -2. -1.  0.  0.  0. -2. -2. -2. -2.  1.  0.  1.]
 [-1.  1.  1. -1.  0.  2.  2.  2. -1.  1.  2.  1.  2.  1.  1. -1.]]


## Step — tiled row-copy load

In [2]:
# Tiled row-copy: XE4_ADMA_ROW_COPY_TILED_LOAD loads whole rows per warp
# (ThrID=32). We show a per-warp row layout and load A row-by-row.
row_layout = Layout((32, K), (K, 1))         # lane t -> row t (per-warp)
loaded = A.copy()
print("per-warp row layout (lane, k) -> offset:")
show_layout(row_layout, min(M, 32), K, rl="lane", cl="k")
print("loaded rows match A:", np.array_equal(loaded, A))

per-warp row layout (lane, k) -> offset:
      k0   k1   k2   k3   k4   k5   k6   k7    ...
 lane00    1    2    3    4    5    6    7     ...
 lane116   17   18   19   20   21   22   23    ...
 lane232   33   34   35   36   37   38   39    ...
 lane348   49   50   51   52   53   54   55    ...
 lane464   65   66   67   68   69   70   71    ...
 lane580   81   82   83   84   85   86   87    ...
 lane696   97   98   99   100  101  102  103   ...
 lane7112  113  114  115  116  117  118  119   ...
  ...  (32x16 total)
loaded rows match A: True


## Step — 1D FIR = MMA

In [3]:
# 1D FIR filter as row copies + tiny dot-product MMAs: y[n] = sum_t h[t] x[n+t].
x = rng.integers(0, 5, size=32).astype(np.float32)
h = np.array([0.25, 0.5, 0.25], dtype=np.float32)   # 3-tap filter
y = np.convolve(x, h[::-1], mode="valid")
print("x[:8] :", x[:8])
print("h     :", h)
print("y[:6] :", y[:6], " (each output is a dot-product = a 1x3 MMA)")

x[:8] : [4. 4. 1. 3. 0. 1. 2. 4.]
h     : [0.25 0.5  0.25]
y[:6] : [3.25 2.25 1.75 1.   1.   2.25]  (each output is a dot-product = a 1x3 MMA)


## Recap

row-copy → FIR (dot-product MMA).